# Lora 实战

## Step1 导入相关包

In [1]:
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, DataCollatorForSeq2Seq, TrainingArguments, Trainer

/home/haoyu/anaconda3/envs/INT8/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Step2 加载数据集

In [2]:
ds = Dataset.load_from_disk("data/alpaca_data_zh/")
ds

Dataset({
    features: ['output', 'input', 'instruction'],
    num_rows: 26858
})

In [3]:
# ds是Dataset类型对象，支持通过索引（如ds[0]）直接访问数据集中的单个样本。这是因为Dataset类实现了类似Python列表的接口，使得它既高效又易于使用。
# Dataset类实现了Python的魔法方法__getitem__。这使得你可以使用索引操作来访问数据集中的元素。
# 当你执行ds[0]时，实际上是调用了Dataset类内部的__getitem__方法来获取第一个样本。
print(type(ds))
ds[0]

<class 'datasets.arrow_dataset.Dataset'>


{'output': '以下是保持健康的三个提示：\n\n1. 保持身体活动。每天做适当的身体运动，如散步、跑步或游泳，能促进心血管健康，增强肌肉力量，并有助于减少体重。\n\n2. 均衡饮食。每天食用新鲜的蔬菜、水果、全谷物和脂肪含量低的蛋白质食物，避免高糖、高脂肪和加工食品，以保持健康的饮食习惯。\n\n3. 睡眠充足。睡眠对人体健康至关重要，成年人每天应保证 7-8 小时的睡眠。良好的睡眠有助于减轻压力，促进身体恢复，并提高注意力和记忆力。',
 'input': '',
 'instruction': '保持健康的三个提示。'}

In [4]:
print(len("以下是保持健康的三个提示：\n\n1. 保持身体活动。每天做适当的身体运动，如散步、跑步或游泳，能促进心血管健康，增强肌肉力量，并有助于减少体重。\n\n2. 均衡饮食。每天食用新鲜的蔬菜、水果、全谷物和脂肪含量低的蛋白质食物，避免高糖、高脂肪和加工食品，以保持健康的饮食习惯。\n\n3. 睡眠充足。睡眠对人体健康至关重要，成年人每天应保证 7-8 小时的睡眠。良好的睡眠有助于减轻压力，促进身体恢复，并提高注意力和记忆力。"))

207


## Step3 数据集预处理

In [5]:
tokenizer = AutoTokenizer.from_pretrained("/media/haoyu/Repo/model/llama-3-chinese-8b-instruct-v2")
tokenizer

PreTrainedTokenizerFast(name_or_path='/media/haoyu/Repo/model/llama-3-chinese-8b-instruct-v2', vocab_size=128000, model_max_length=1000000000000000019884624838656, is_fast=True, padding_side='right', truncation_side='right', special_tokens={'bos_token': '<|begin_of_text|>', 'eos_token': '<|end_of_text|>', 'pad_token': '<|end_of_text|>'}, clean_up_tokenization_spaces=True, added_tokens_decoder={
	128000: AddedToken("<|begin_of_text|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	128001: AddedToken("<|end_of_text|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	128002: AddedToken("<|reserved_special_token_0|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	128003: AddedToken("<|reserved_special_token_1|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	128004: AddedToken("<|reserved_special_token_2|>", rstrip=False, lstrip=False, single_word=False,

In [6]:
tokenizer.padding_side = "right"  # 一定要设置padding_side为right，否则batch大于1时可能不收敛

In [7]:
tokenizer.pad_token_id = 2  # 'pad_token': '<|end_of_text|>'

In [8]:
def process_func(example):
    MAX_LENGTH = 384    # Llama分词器会将一个中文字切分为多个token，因此需要放开一些最大长度，保证数据的完整性
    input_ids, attention_mask, labels = [], [], []

    # 调用join函数将["Human: " + example["instruction"], example["input"]]3个字符串连接在一起，调用strip函数去掉字符串头尾的空格。
    # 再连接字符串"\n\nAssistant: "。最后调用tokenizer进行向量化。
    # 最终构建了指令微调中的指令（instruction）。
    instruction = tokenizer("\n".join(["Human: " + example["instruction"], example["input"]]).strip() + "\n\nAssistant: ", add_special_tokens=False)
    
    # 构建指令微调中的回答（response），即对example["output"]调用tokenizer进行向量化。
    response = tokenizer(example["output"], add_special_tokens=False)

    # 提取instruction和response的input_ids，加上eos_token_id，构建input_ids。
    input_ids = instruction["input_ids"] + response["input_ids"] + [tokenizer.eos_token_id]

    # 提取instruction和response的attention_mask，加上eos_token_id对应的maks"1"，构建attention_mask。
    attention_mask = instruction["attention_mask"] + response["attention_mask"] + [1]

    # 在序列生成任务中，我们需要告诉模型哪些部分是输入（不需要预测）以及哪些部分是输出（需要预测）。通常的做法是：
    # 对于输入部分（如instruction），我们将其标记为特殊的填充值（如-100），表示这些位置的token不参与损失计算。
    # 对于输出部分（如response），我们保留其实际的token ID，表示这些位置的token是模型需要预测的目标。
    # [-100] * len(instruction["input_ids"])创建了一个与输入部分等长的列表，其中每个元素都是 -100。
    labels = [-100] * len(instruction["input_ids"]) + response["input_ids"] + [tokenizer.eos_token_id]

    # 以上就创建了input_ids，attention_mask，labels三个变量（列表）。
    # 如果列表长度大于MAX_LENGTH，则进行截断，只保留前MAX_LENGTH个元素。
    if len(input_ids) > MAX_LENGTH:
        input_ids = input_ids[:MAX_LENGTH]
        attention_mask = attention_mask[:MAX_LENGTH]
        labels = labels[:MAX_LENGTH]
    
    # 返回值是由input_ids，attention_mask，labels三个变量（列表）组成的字典。
    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels
    }

In [9]:
ds.column_names

['output', 'input', 'instruction']

In [10]:
# map方法的核心功能是对数据集中的每个样本应用一个指定的函数（如process_func），并返回一个新的Dataset对象。
# 为什么ds可以使用map方法？
# 因为Dataset类内部实现了map方法，使得它可以像其他容器类型（如列表）一样支持高阶函数操作。具体来说，map方法会遍历数据集中的每个样本，并将指定的函数应用到每个样本上。
# 惰性计算（Lazy Evaluation）
# map方法不会立即执行所有的转换操作，而是采用惰性计算（lazy evaluation）的方式。这意味着只有在实际需要访问数据时（例如迭代或保存数据集时），才会真正执行process_func。
# remove_columns=ds.column_names: 表示在生成的新数据集中移除原始列，因为这些列已经被处理并替换为新的列。

# 所以，这里调用map方法的作用是对数据集进行预处理，例如执行tokenizer。
tokenized_ds = ds.map(process_func, remove_columns=ds.column_names)
tokenized_ds

Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 26858
})

In [11]:
# tokenized_ds还是一个Dataset类对象，所以仍然可以使用索引（如tokenized_ds[0]）直接访问数据集中的单个样本。
print(tokenized_ds[0]["input_ids"])

[35075, 25, 111505, 69978, 113614, 9554, 126524, 46239, 3490, 72803, 25, 220, 88852, 21043, 118551, 113614, 9554, 126524, 46239, 49543, 16, 13, 111505, 69978, 111006, 108726, 1811, 74257, 36827, 102210, 108562, 40265, 9554, 111006, 114253, 116051, 107471, 65782, 5486, 110774, 65782, 58291, 83994, 126503, 3922, 27327, 113096, 42399, 64209, 104473, 36651, 113614, 3922, 50285, 103229, 120044, 107079, 120772, 91495, 19361, 103129, 35304, 111689, 83747, 33014, 30358, 3490, 17, 13, 111020, 229, 120383, 120522, 102456, 1811, 74257, 36827, 102456, 11883, 17039, 118882, 9554, 107139, 105, 108171, 5486, 53610, 28873, 5486, 37087, 104858, 53953, 34208, 121496, 57942, 103, 96412, 33857, 103167, 9554, 111678, 101828, 103706, 102456, 53953, 3922, 111098, 103048, 45736, 117587, 122603, 121496, 57942, 103, 34208, 117041, 119008, 105610, 118551, 113614, 9554, 120522, 102456, 105369, 33565, 107, 3490, 18, 13, 124022, 94, 120379, 105843, 102780, 1811, 113136, 120379, 33764, 17792, 33014, 113614, 57237, 3

In [12]:
print(tokenized_ds[0]["attention_mask"])

[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]


In [13]:
print(tokenized_ds[0]["labels"])

[-100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, 88852, 21043, 118551, 113614, 9554, 126524, 46239, 49543, 16, 13, 111505, 69978, 111006, 108726, 1811, 74257, 36827, 102210, 108562, 40265, 9554, 111006, 114253, 116051, 107471, 65782, 5486, 110774, 65782, 58291, 83994, 126503, 3922, 27327, 113096, 42399, 64209, 104473, 36651, 113614, 3922, 50285, 103229, 120044, 107079, 120772, 91495, 19361, 103129, 35304, 111689, 83747, 33014, 30358, 3490, 17, 13, 111020, 229, 120383, 120522, 102456, 1811, 74257, 36827, 102456, 11883, 17039, 118882, 9554, 107139, 105, 108171, 5486, 53610, 28873, 5486, 37087, 104858, 53953, 34208, 121496, 57942, 103, 96412, 33857, 103167, 9554, 111678, 101828, 103706, 102456, 53953, 3922, 111098, 103048, 45736, 117587, 122603, 121496, 57942, 103, 34208, 117041, 119008, 105610, 118551, 113614, 9554, 120522, 102456, 105369, 33565, 107, 3490, 18, 13, 124022, 94, 120379, 105843, 102780, 1811, 113136, 120379, 33764, 17792, 33014, 113614, 57237, 30356,

In [14]:
tokenizer("abc " + tokenizer.eos_token)

{'input_ids': [128000, 13997, 220, 128001], 'attention_mask': [1, 1, 1, 1]}

In [15]:
tokenizer.decode(tokenized_ds[0]["input_ids"])

'Human: 保持健康的三个提示。\n\nAssistant: 以下是保持健康的三个提示：\n\n1. 保持身体活动。每天做适当的身体运动，如散步、跑步或游泳，能促进心血管健康，增强肌肉力量，并有助于减少体重。\n\n2. 均衡饮食。每天食用新鲜的蔬菜、水果、全谷物和脂肪含量低的蛋白质食物，避免高糖、高脂肪和加工食品，以保持健康的饮食习惯。\n\n3. 睡眠充足。睡眠对人体健康至关重要，成年人每天应保证 7-8 小时的睡眠。良好的睡眠有助于减轻压力，促进身体恢复，并提高注意力和记忆力。<|end_of_text|>'

In [16]:
tokenizer.decode(list(filter(lambda x: x != -100, tokenized_ds[1]["labels"])))

'4/16等于1/4是因为我们可以约分分子分母都除以他们的最大公约数4，得到（4÷4）/ (16÷4）=1/4。分数的约分是用分子和分母除以相同的非零整数，来表示分数的一个相同的值，这因为分数实际上表示了分子除以分母，所以即使两个数同时除以同一个非零整数，分数的值也不会改变。所以4/16 和1/4是两种不同的书写形式，但它们的值相等。<|end_of_text|>'

## Step4 创建模型

In [17]:
import torch
from transformers import BitsAndBytesConfig

# 多卡情况，可以去掉device_map="auto"，否则会将模型拆开
"""
model = AutoModelForCausalLM.from_pretrained("/media/haoyu/Repo/model/llama-3-chinese-8b-instruct-v2", low_cpu_mem_usage=True, 
                                             torch_dtype=torch.bfloat16, device_map="auto", load_in_4bit=True, bnb_4bit_compute_dtype=torch.bfloat16,
                                             bnb_4bit_quant_type="nf4", bnb_4bit_use_double_quant=True)
"""

# 使用BitsAndBytes INT4量化导入模型
bnb_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_quant_type="nf4", bnb_4bit_use_double_quant=True)
model = AutoModelForCausalLM.from_pretrained("/media/haoyu/Repo/model/llama-3-chinese-8b-instruct-v2", low_cpu_mem_usage=True, 
                                             torch_dtype=torch.bfloat16, device_map="auto", quantization_config=bnb_config)

Loading checkpoint shards: 100%|██████████| 4/4 [00:04<00:00,  1.14s/it]


In [18]:
for name, param in model.named_parameters():
    print(name, param.shape, param.dtype)

model.embed_tokens.weight torch.Size([128256, 4096]) torch.bfloat16
model.layers.0.self_attn.q_proj.weight torch.Size([8388608, 1]) torch.uint8
model.layers.0.self_attn.k_proj.weight torch.Size([2097152, 1]) torch.uint8
model.layers.0.self_attn.v_proj.weight torch.Size([2097152, 1]) torch.uint8
model.layers.0.self_attn.o_proj.weight torch.Size([8388608, 1]) torch.uint8
model.layers.0.mlp.gate_proj.weight torch.Size([29360128, 1]) torch.uint8
model.layers.0.mlp.up_proj.weight torch.Size([29360128, 1]) torch.uint8
model.layers.0.mlp.down_proj.weight torch.Size([29360128, 1]) torch.uint8
model.layers.0.input_layernorm.weight torch.Size([4096]) torch.bfloat16
model.layers.0.post_attention_layernorm.weight torch.Size([4096]) torch.bfloat16
model.layers.1.self_attn.q_proj.weight torch.Size([8388608, 1]) torch.uint8
model.layers.1.self_attn.k_proj.weight torch.Size([2097152, 1]) torch.uint8
model.layers.1.self_attn.v_proj.weight torch.Size([2097152, 1]) torch.uint8
model.layers.1.self_attn.o_

In [19]:
model.config

LlamaConfig {
  "_attn_implementation_autoset": true,
  "_name_or_path": "/media/haoyu/Repo/model/llama-3-chinese-8b-instruct-v2",
  "architectures": [
    "LlamaForCausalLM"
  ],
  "attention_bias": false,
  "attention_dropout": 0.0,
  "bos_token_id": 128000,
  "eos_token_id": 128001,
  "head_dim": 128,
  "hidden_act": "silu",
  "hidden_size": 4096,
  "initializer_range": 0.02,
  "intermediate_size": 14336,
  "max_position_embeddings": 8192,
  "mlp_bias": false,
  "model_type": "llama",
  "num_attention_heads": 32,
  "num_hidden_layers": 32,
  "num_key_value_heads": 8,
  "pretraining_tp": 1,
  "quantization_config": {
    "_load_in_4bit": true,
    "_load_in_8bit": false,
    "bnb_4bit_compute_dtype": "bfloat16",
    "bnb_4bit_quant_storage": "uint8",
    "bnb_4bit_quant_type": "nf4",
    "bnb_4bit_use_double_quant": true,
    "llm_int8_enable_fp32_cpu_offload": false,
    "llm_int8_has_fp16_weight": false,
    "llm_int8_skip_modules": null,
    "llm_int8_threshold": 6.0,
    "load_in

## Lora

### PEFT Step1 配置文件

In [20]:
from peft import LoraConfig, TaskType, get_peft_model

# 创建LoRA配置
config = LoraConfig(task_type=TaskType.CAUSAL_LM,)
config

LoraConfig(task_type=<TaskType.CAUSAL_LM: 'CAUSAL_LM'>, peft_type=<PeftType.LORA: 'LORA'>, auto_mapping=None, base_model_name_or_path=None, revision=None, inference_mode=False, r=8, target_modules=None, exclude_modules=None, lora_alpha=8, lora_dropout=0.0, fan_in_fan_out=False, bias='none', use_rslora=False, modules_to_save=None, init_lora_weights=True, layers_to_transform=None, layers_pattern=None, rank_pattern={}, alpha_pattern={}, megatron_config=None, megatron_core='megatron.core', trainable_token_indices=None, loftq_config={}, eva_config=None, corda_config=None, use_dora=False, layer_replication=None, runtime_config=LoraRuntimeConfig(ephemeral_gpu_offload=False), lora_bias=False)

### PEFT Step2 创建模型

In [21]:
# 合并LoRA与模型
model = get_peft_model(model, config)

In [22]:
# config增加了模型信息
config

LoraConfig(task_type=<TaskType.CAUSAL_LM: 'CAUSAL_LM'>, peft_type=<PeftType.LORA: 'LORA'>, auto_mapping=None, base_model_name_or_path='/media/haoyu/Repo/model/llama-3-chinese-8b-instruct-v2', revision=None, inference_mode=False, r=8, target_modules={'q_proj', 'v_proj'}, exclude_modules=None, lora_alpha=8, lora_dropout=0.0, fan_in_fan_out=False, bias='none', use_rslora=False, modules_to_save=None, init_lora_weights=True, layers_to_transform=None, layers_pattern=None, rank_pattern={}, alpha_pattern={}, megatron_config=None, megatron_core='megatron.core', trainable_token_indices=None, loftq_config={}, eva_config=None, corda_config=None, use_dora=False, layer_replication=None, runtime_config=LoraRuntimeConfig(ephemeral_gpu_offload=False), lora_bias=False)

In [23]:
model.enable_input_require_grads() # 开启梯度检查点时，要执行该方法

In [24]:
# model = model.half()  # 当整个模型都是半精度时，需要将adam_epsilon调大
# torch.tensor(1e-8).half() 

In [25]:
model.print_trainable_parameters()

trainable params: 3,407,872 || all params: 8,033,669,120 || trainable%: 0.0424


## Step5 配置训练参数

In [26]:
args = TrainingArguments(
    output_dir="./chatbot",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=32,
    logging_steps=10,
    num_train_epochs=1,
    gradient_checkpointing=True,
    optim="paged_adamw_32bit"
)

## Step6 创建训练器

In [27]:
trainer = Trainer(
    model=model,
    args=args,
    tokenizer=tokenizer,
    train_dataset=tokenized_ds.select(range(6000)),
    data_collator=DataCollatorForSeq2Seq(tokenizer=tokenizer, padding=True),
)

[2025-03-24 12:07:52,531] [INFO] [real_accelerator.py:222:get_accelerator] Setting ds_accelerator to cuda (auto detect)


/tmp/ipykernel_107695/2462369116.py:1: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
/home/haoyu/anaconda3/envs/INT8/compiler_compat/ld: warning: librt.so.1, needed by /usr/local/cuda-12.6/lib64/libcufile.so, not found (try using -rpath or -rpath-link)
/home/haoyu/anaconda3/envs/INT8/compiler_compat/ld: warning: libpthread.so.0, needed by /usr/local/cuda-12.6/lib64/libcufile.so, not found (try using -rpath or -rpath-link)
/home/haoyu/anaconda3/envs/INT8/compiler_compat/ld: warning: libstdc++.so.6, needed by /usr/local/cuda-12.6/lib64/libcufile.so, not found (try using -rpath or -rpath-link)
/home/haoyu/anaconda3/envs/INT8/compiler_compat/ld: warning: libm.so.6, needed by /usr/local/cuda-12.6/lib64/libcufile.so, not found (try using -rpath or -rpath-link)
/home/haoyu/anaconda3/envs/INT8/compiler_compat/ld: /usr/local/cuda-12.6/lib64/libcufile.so: undefined reference to `std::runt

## Step7 模型训练

In [28]:
trainer.train()

`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


Step,Training Loss
10,1.566400
20,1.534500
30,1.457000
40,1.501000
50,1.350800
60,1.332300
70,1.292100
80,1.321000
90,1.323600
100,1.322700


TrainOutput(global_step=187, training_loss=1.3477325592449005, metrics={'train_runtime': 2241.5819, 'train_samples_per_second': 2.677, 'train_steps_per_second': 0.083, 'total_flos': 2.488277923413197e+16, 'train_loss': 1.3477325592449005, 'epoch': 0.9973333333333333})

## Step8 模型推理

In [29]:
model.eval()
ipt = tokenizer("Human: {}\n{}".format("你好", "").strip() + "\n\nAssistant: ", return_tensors="pt").to(model.device)
tokenizer.decode(model.generate(**ipt, max_length=128, do_sample=True, eos_token_id=tokenizer.eos_token_id)[0], skip_special_tokens=True)

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


'Human: 你好\n\nAssistant: 你好！很高兴为你服务。有什么我可以帮助你的吗？'

In [30]:
model.merge_and_unload() 

/home/haoyu/anaconda3/envs/INT8/lib/python3.12/site-packages/peft/tuners/lora/bnb.py:351: UserWarning: Merge lora module to 4-bit linear may get different generations due to rounding errors.
  warnings.warn(


LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(128256, 4096)
    (layers): ModuleList(
      (0-31): 32 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear4bit(in_features=4096, out_features=4096, bias=False)
          (k_proj): Linear4bit(in_features=4096, out_features=1024, bias=False)
          (v_proj): Linear4bit(in_features=4096, out_features=1024, bias=False)
          (o_proj): Linear4bit(in_features=4096, out_features=4096, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear4bit(in_features=4096, out_features=14336, bias=False)
          (up_proj): Linear4bit(in_features=4096, out_features=14336, bias=False)
          (down_proj): Linear4bit(in_features=14336, out_features=4096, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): LlamaRMSNorm((4096,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((4096,), eps=1e-05)
      )
    )
    (norm): LlamaRMSNorm((409